# Bounding-box object insertion with FLUX.2 [klein] 4B (v2)

**bbox → inpaint → pick best candidate → adaptive blend.**

Klein has no native bbox input and treats the box as a *loose hint*, so v2 handles that explicitly:

| Problem seen in v1 | Fix in v2 |
|---|---|
| Object overflows the box and gets sliced off by the hard mask | **Change-aware mask:** the blend region is the box ∪ everything the model actually changed that touches the box, so overflow is kept, not clipped |
| Some seeds overflow badly or don't insert anything | **Best-of-N:** generate N candidates and automatically pick the one with the least overflow / no magenta leftovers |
| Visible gray rectangle around the object | **Poisson (gradient-domain) blend** at the region border instead of a plain alpha fade |
| Object looks too crisp / rendered vs. low-quality source | Prompt asks for matching quality + **auto sharpness matching** (+ optional JPEG round-trip); working resolution follows the crop's native size |
| Object bigger / different shape than the box | Prompt includes box shape and "fully inside the rectangle with a margin" |

Pixels outside the final blend region are **bit-identical** to the input.

In [ ]:
import torch
from diffusers import Flux2KleinPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"

# Distilled 4B: 4 steps, guidance 1.0 (Apache 2.0, ~13 GB VRAM)
pipe = Flux2KleinPipeline.from_pretrained("black-forest-labs/FLUX.2-klein-4B", torch_dtype=torch.bfloat16)

pipe.to(device)                        # fastest, needs ~13+ GB VRAM
# pipe.enable_model_cpu_offload()      # use this instead if you run out of memory

## Helpers

In [ ]:
import io
import numpy as np
import cv2
from scipy import ndimage as ndi
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import torch

MASK_COLOR = (255, 0, 255)  # magenta placeholder


# ---------- geometry ----------
def get_crop_box(bbox, img_size, context_scale=2.5, min_side=256):
    """Crop around bbox: bbox size * context_scale per axis, clamped to the image."""
    x0, y0, x1, y1 = bbox
    W, H = img_size
    w, h = x1 - x0, y1 - y0
    cw = min(W, max(w * context_scale, min_side))
    ch = min(H, max(h * context_scale, min_side))
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    left = min(max(cx - cw / 2, 0), W - cw)
    top = min(max(cy - ch / 2, 0), H - ch)
    return tuple(int(round(v)) for v in (left, top, left + cw, top + ch))


def work_size(w, h, target):
    """Scale so the long side is ~target; both sides multiples of 16."""
    s = target / max(w, h)
    return max(16, round(w * s / 16) * 16), max(16, round(h * s / 16) * 16)


def auto_target(cw, ch, lo=512, hi=1024):
    """Work near the crop's native resolution (avoids over-sharp 'rendered' look)."""
    return int(min(hi, max(lo, max(cw, ch))))


def box_mask_np(shape_hw, box):
    h, w = shape_hw
    m = np.zeros((h, w), bool)
    x0, y0, x1, y1 = [int(round(v)) for v in box]
    m[max(y0, 0):min(y1, h), max(x0, 0):min(x1, w)] = True
    return m


# ---------- prompting ----------
def default_prompt(obj, box_w, box_h, has_reference):
    ar = box_w / box_h
    shape = "wide, landscape-shaped" if ar > 1.3 else ("tall, portrait-shaped" if ar < 0.77 else "roughly square")
    quality = ("Match the lighting, colors, motion blur, resolution and compression quality of the surrounding image. "
               "No drop shadow, no studio look. Keep everything else unchanged.")
    if has_reference:
        return (f"Replace the solid magenta rectangle in the first image with the {obj} from the second image. "
                f"The rectangle is {shape}; the whole {obj} must fit completely inside it with a small margin and "
                f"must not extend beyond its edges. {quality}")
    return (f"Replace the solid magenta rectangle with {obj}. The rectangle is {shape}; the whole {obj} must fit "
            f"completely inside it with a small margin and must not extend beyond its edges. {quality}")


# ---------- candidate analysis ----------
def analyze_candidate(out, work, wbox, thr=28):
    """Compare model output with the (unmasked) original at working resolution."""
    o = np.asarray(out, np.int16)
    w = np.asarray(work, np.int16)
    diff = np.abs(o - w).max(2).astype(np.float32)
    diff = cv2.GaussianBlur(diff, (0, 0), 2)                      # suppress VAE noise
    changed = ndi.binary_opening(diff > thr, iterations=2)
    bm = box_mask_np(diff.shape, wbox)
    area = max(bm.sum(), 1)
    overflow = (changed & ~bm).sum() / area                        # changed pixels outside the box
    inside_change = (changed & bm).sum() / area                    # did anything get inserted?
    mag = ((np.abs(o - np.array(MASK_COLOR)).sum(2) < 90) & bm).sum() / area  # magenta leftovers
    score = overflow + 3.0 * mag + (1.0 if inside_change < 0.05 else 0.0)
    return dict(score=float(score), overflow=float(overflow), inside=float(inside_change),
                magenta=float(mag), changed=changed, box=bm)


def build_region(changed, bm, grow=6, max_growth=3.0):
    """Blend region = box + changed pixels connected to it (keeps overflow), then dilated."""
    region = changed | bm
    lab, _ = ndi.label(region)
    keep = np.unique(lab[bm]); keep = keep[keep > 0]
    region = np.isin(lab, keep)
    if region.sum() > max_growth * bm.sum():                       # global drift guard
        region = bm.copy()
    return ndi.binary_dilation(region, iterations=grow)


# ---------- blending / matching ----------
def poisson_blend(src, dst, region):
    """Gradient-domain blend of src into dst inside bool `region` (all at the same size, RGB uint8)."""
    pad = 32
    s = cv2.copyMakeBorder(src, pad, pad, pad, pad, cv2.BORDER_REFLECT)
    d = cv2.copyMakeBorder(dst, pad, pad, pad, pad, cv2.BORDER_REFLECT)
    m = cv2.copyMakeBorder(region.astype(np.uint8) * 255, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=0)
    ys, xs = np.where(m > 0)
    center = (int((xs.min() + xs.max()) // 2), int((ys.min() + ys.max()) // 2))
    res = cv2.seamlessClone(s, d, m, center, cv2.NORMAL_CLONE)
    return res[pad:-pad, pad:-pad]


def _lap_var(gray, mask):
    return float(cv2.Laplacian(gray, cv2.CV_32F)[mask].var())


def match_sharpness(gen, ref, region, max_blur=2.0):
    """Blur `gen` slightly so its local detail level matches the surrounding original."""
    ring = ndi.binary_dilation(region, iterations=40) & ~ndi.binary_dilation(region, iterations=4)
    if ring.sum() < 100:
        return gen, 0.0
    g_ref = cv2.cvtColor(ref, cv2.COLOR_RGB2GRAY)
    g_gen = cv2.cvtColor(gen, cv2.COLOR_RGB2GRAY)
    target = np.log(_lap_var(g_ref, ring) + 1e-6)
    sigmas = [s for s in (0, 0.5, 0.8, 1.2, 1.6, 2.0, 2.5, 3.0) if s <= max_blur]
    def err(s):
        g = cv2.GaussianBlur(g_gen, (0, 0), s) if s > 0 else g_gen
        return abs(np.log(_lap_var(g, region) + 1e-6) - target)
    best = min(sigmas, key=err)
    return (cv2.GaussianBlur(gen, (0, 0), best) if best > 0 else gen), best


def jpeg_roundtrip(arr, q):
    buf = io.BytesIO()
    Image.fromarray(arr).save(buf, "JPEG", quality=q)
    return np.asarray(Image.open(buf).convert("RGB"))


# ---------- main ----------
@torch.no_grad()
def insert_object(
    pipe, image, bbox, obj, reference=None, prompt=None,
    n_candidates=6, seed=0,
    context_scale=2.5, target=None,
    steps=4, guidance=1.0,
    thr=28, grow=6, feather=8, max_growth=3.0,
    blend="poisson",            # "poisson" (hides the box seam) or "alpha"
    match_quality=True, max_blur=2.0, jpeg_quality=None,
):
    """
    image : PIL image;  bbox : (x0, y0, x1, y1) pixels;  obj : e.g. "a red ceramic teapot"
    reference : optional PIL image of the object to insert
    Returns (final_image, debug_dict)
    """
    image = image.convert("RGB")
    x0, y0, x1, y1 = bbox
    assert 0 <= x0 < x1 <= image.width and 0 <= y0 < y1 <= image.height, "bbox outside image"

    # 1) crop with context, resize to a Flux-friendly size near native resolution
    crop_box = get_crop_box(bbox, image.size, context_scale)
    crop = image.crop(crop_box)
    cw, ch = crop.size
    ww, wh = work_size(cw, ch, target or auto_target(cw, ch))
    sx, sy = ww / cw, wh / ch
    wbox = ((x0 - crop_box[0]) * sx, (y0 - crop_box[1]) * sy,
            (x1 - crop_box[0]) * sx, (y1 - crop_box[1]) * sy)
    work = crop.resize((ww, wh), Image.LANCZOS)

    # 2) paint the bbox
    masked = work.copy()
    ImageDraw.Draw(masked).rectangle(wbox, fill=MASK_COLOR)
    refs = [masked] + ([reference.convert("RGB")] if reference is not None else [])
    prompt = prompt or default_prompt(obj, x1 - x0, y1 - y0, reference is not None)

    # 3) best-of-N
    cands = []
    gdev = pipe.device if hasattr(pipe, "device") else "cpu"
    for i in range(n_candidates):
        gen = torch.Generator(device=gdev).manual_seed(seed + i)
        out = pipe(image=refs if len(refs) > 1 else refs[0], prompt=prompt, height=wh, width=ww,
                   num_inference_steps=steps, guidance_scale=guidance, generator=gen).images[0].convert("RGB")
        if out.size != (ww, wh):
            out = out.resize((ww, wh), Image.LANCZOS)
        a = analyze_candidate(out, work, wbox, thr)
        a.update(seed=seed + i, out=out)
        cands.append(a)
    best = min(cands, key=lambda c: c["score"])
    out = best["out"]

    # 4) change-aware region (keeps overflow), then to full crop resolution
    region_w = build_region(best["changed"], best["box"], grow, max_growth)
    region = cv2.resize(region_w.astype(np.uint8), (cw, ch), interpolation=cv2.INTER_NEAREST).astype(bool)
    gen_c = np.asarray(out.resize((cw, ch), Image.LANCZOS))
    crop_np = np.asarray(crop)

    # 5) match source quality
    blur_used = 0.0
    if match_quality:
        gen_c, blur_used = match_sharpness(gen_c, crop_np, region, max_blur)
    if jpeg_quality:
        gen_c = jpeg_roundtrip(gen_c, jpeg_quality)

    # 6) blend
    if blend == "poisson":
        try:
            new_crop = poisson_blend(gen_c, crop_np, region)
        except Exception as e:
            print("poisson failed, falling back to alpha:", e)
            blend = "alpha"
    if blend == "alpha":
        alpha = cv2.GaussianBlur(region.astype(np.float32), (0, 0), max(feather / 2, 0.1))[..., None]
        new_crop = (gen_c * alpha + crop_np * (1 - alpha)).round().astype(np.uint8)

    new_crop = np.where(ndi.binary_dilation(region, iterations=feather + 2)[..., None], new_crop, crop_np)
    final = image.copy()
    final.paste(Image.fromarray(new_crop.astype(np.uint8)), crop_box[:2])

    dbg = dict(prompt=prompt, crop_box=crop_box, masked_input=masked, raw_output=out, region=region,
               candidates=[{k: c[k] for k in ("seed", "score", "overflow", "inside", "magenta", "out")} for c in cands],
               chosen_seed=best["seed"], blur_used=blur_used)
    return final, dbg


# ---------- display ----------
def show(images, titles=None, figsize=(16, 6)):
    fig, axs = plt.subplots(1, len(images), figsize=figsize)
    axs = [axs] if len(images) == 1 else axs
    for i, (ax, im) in enumerate(zip(axs, images)):
        ax.imshow(im, cmap="gray" if getattr(im, "mode", "") == "L" else None)
        ax.axis("off")
        if titles: ax.set_title(titles[i])
    plt.tight_layout(); plt.show()


def show_candidates(dbg, cols=6):
    cs = dbg["candidates"]
    cols = min(cols, len(cs)); rows = (len(cs) + cols - 1) // cols
    fig, axs = plt.subplots(rows, cols, figsize=(3.2 * cols, 2.6 * rows), squeeze=False)
    for ax in axs.ravel(): ax.axis("off")
    for ax, c in zip(axs.ravel(), cs):
        ax.imshow(c["out"])
        chosen = c["seed"] == dbg["chosen_seed"]
        ax.set_title(f"seed {c['seed']}  score {c['score']:.2f}" + ("  ✓" if chosen else ""),
                     fontsize=9, color="green" if chosen else "black")
    plt.tight_layout(); plt.show()


def preview_bbox(image, bbox, color="red", width=4):
    im = image.convert("RGB").copy()
    ImageDraw.Draw(im).rectangle(bbox, outline=color, width=width)
    return im


def norm_to_px(bbox_norm, size):
    W, H = size
    return (bbox_norm[0] * W, bbox_norm[1] * H, bbox_norm[2] * W, bbox_norm[3] * H)

## Load your image and set the bounding box

`bbox` is `(x0, y0, x1, y1)` in **pixels** (use `norm_to_px` for normalized boxes).

In [ ]:
# image = load_image("https://.../your.png")        # from diffusers.utils import load_image
image = Image.open("scene.jpg").convert("RGB")     # <-- your image
bbox = (400, 300, 700, 600)                        # <-- your box
# bbox = norm_to_px((0.4, 0.5, 0.6, 0.8), image.size)

show([preview_bbox(image, bbox)], ["Input + bbox"], figsize=(8, 8))

## Run

In [ ]:
result, dbg = insert_object(
    pipe, image, bbox,
    obj="a white fish swimming",     # describe the object (and pose/direction if you care)
    n_candidates=6,                  # more = better pick, slower (4-step model, so cheap)
    seed=0,
)

print("Prompt:", dbg["prompt"])
print(f"chosen seed {dbg['chosen_seed']}, sharpness blur sigma {dbg['blur_used']}")
show([image, dbg["masked_input"], dbg["raw_output"], result],
     ["Original", "Model input (crop)", "Chosen raw output (crop)", "Final"])
show_candidates(dbg)
result.save("result.png")

## Optional: insert an object from a reference image

In [ ]:
# reference = Image.open("object.png").convert("RGB")   # object on a plain background works best
# result, dbg = insert_object(pipe, image, bbox, obj="the fish", reference=reference)
# show([image, result], ["Original", "Final"])

## Tuning guide

| Symptom | Knob |
|---|---|
| Object still looks too crisp / CG | `max_blur=3.0`, `jpeg_quality=60`, or lower `target` (e.g. 512) |
| Color tint on the object after blending | `blend="alpha"` |
| Visible seam / gray patch | keep `blend="poisson"`, raise `grow` (10–16) |
| Object bigger than your box | tighten the box; raise `n_candidates` (overflow is scored, the lowest wins) |
| Object not inserted / magenta left | raise `n_candidates`; change `MASK_COLOR` (e.g. `(0,255,0)`) and the word "magenta" in the prompt |
| Region grows to cover too much | lower `max_growth` (e.g. 2.0) or raise `thr` (e.g. 40) |
| Wrong pose / direction / motion blur | describe it in `obj` ("a white fish swimming left, slightly motion-blurred"); a box can't carry pose |
| `guidance_scale` rejected | remove it. For the *base* model use `steps=50, guidance=4.0` |